3. 🏁 Modeling (Training, tuning, comparison)

Phase 3: Modeling

Time-Series Cross-Validation: Validated model stability across sequential time folds to prove the AI's predictions are consistent and not reliant on lucky data samples.

LightGBM (Quantile Regression): Trained an efficient gradient-boosting model to predict the realistic median (P50) and a conservative safety boundary (P90).

Isolation Forest: Deployed an unsupervised anomaly detection algorithm to scan for statistically "weird" shipments (black swan events) and flag them for massive manual safety buffers.

Survival Analysis (Weibull AFT): Framed logistics as a "time-to-event" problem, training a model to calculate the rigorous mathematical probability of delivery across a time curve.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lifelines import WeibullAFTFitter
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
import os
import joblib

def cross_validate_p50_model(train_df, features, target_col, n_splits=5):
    """
    Performs Time-Series Cross-Validation on the training data to prove
    predictive stability before the final models are trained.
    """
    print(f"\n--- Running {n_splits}-Fold Time-Series Cross-Validation (P50 Baseline) ---")

    X = train_df[features].reset_index(drop=True)
    y = train_df[target_col].reset_index(drop=True)

    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_scores = []

    for fold, (train_index, val_index) in enumerate(tscv.split(X)):
        X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
        y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

        model = lgb.LGBMRegressor(
            objective='quantile',
            alpha=0.5,
            n_estimators=100,
            random_state=42,
            verbose=-1
        )
        model.fit(X_train_fold, y_train_fold)

        preds = model.predict(X_val_fold)
        mae = mean_absolute_error(y_val_fold, preds)
        fold_scores.append(mae)

        print(f" Fold {fold + 1}: Mean Absolute Error = {mae:.2f} days")

    print(f" => Average CV Error: {np.mean(fold_scores):.2f} days (±{np.std(fold_scores):.2f} days)")
    print("-----------------------------------------------------------------------\n")

def train_lgbm_models(X_train, y_train):
    """Trains LightGBM Quantile Regressors (P50 and P90)."""
    print(" - Training LightGBM...")
    model_p50 = lgb.LGBMRegressor(objective='quantile', alpha=0.5, n_estimators=100, random_state=42, verbose=-1)
    model_p90 = lgb.LGBMRegressor(objective='quantile', alpha=0.9, n_estimators=100, random_state=42, verbose=-1)

    model_p50.fit(X_train, y_train)
    model_p90.fit(X_train, y_train)
    return model_p50, model_p90

def train_isolation_forest_pipeline(X_train, y_train):
    """Trains an Isolation Forest for anomaly detection, paired with a standard Random Forest."""
    print(" - Training Isolation Forest (Anomaly Detection)...")

    print("   -> Fitting Base RF Model (Median Target)...")
    rf_p50 = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    rf_p50.fit(X_train, y_train)

    print("   -> Fitting Isolation Forest (Finding Statistical Outliers)...")
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    iso_forest.fit(X_train)

    return rf_p50, iso_forest

def train_survival_model(train_df, features, target_col):
    """Trains a Weibull Accelerated Failure Time (AFT) Survival Model."""
    print(" - Training Survival Analysis (Weibull AFT)...")

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(train_df[features])

    surv_df = pd.DataFrame(scaled_features, columns=features)
    surv_df[target_col] = train_df[target_col].values
    surv_df['Event_Observed'] = 1

    aft = WeibullAFTFitter(penalizer=0.1)
    aft.fit(surv_df, duration_col=target_col, event_col='Event_Observed')

    return aft, scaler

def evaluate_all_models(train_data, test_data, features, target_col):
    """Trains all models, appends their predictions, and packages the objects for export."""
    print("Training Multi-Model Pipeline...")
    X_train, y_train = train_data[features], train_data[target_col]
    X_test = test_data[features]

    # 1. LightGBM
    lgbm_p50, lgbm_p90 = train_lgbm_models(X_train, y_train)
    test_data['Pred_LGBM_P50_Days'] = lgbm_p50.predict(X_test)
    test_data['Pred_LGBM_P90_Days'] = lgbm_p90.predict(X_test)

    # 2. Isolation Forest (Anomaly-based Quoting Hybrid)
    rf_p50, iso_forest = train_isolation_forest_pipeline(X_train, y_train)
    test_data['Pred_IsolationForest_P50_Days'] = rf_p50.predict(X_test)
    anomalies = iso_forest.predict(X_test)
    test_data['Pred_IsolationForest_P90_Days'] = np.where(
        anomalies == -1,
        test_data['Pred_IsolationForest_P50_Days'] + 4.0,
        test_data['Pred_IsolationForest_P50_Days'] + 1.0
    )

    # 3. Survival Analysis
    aft_model, surv_scaler = train_survival_model(train_data, features, target_col)
    X_test_surv = pd.DataFrame(surv_scaler.transform(X_test), columns=features)
    test_data['Pred_Survival_P50_Days'] = aft_model.predict_percentile(X_test_surv, p=0.5)
    test_data['Pred_Survival_P90_Days'] = aft_model.predict_percentile(X_test_surv, p=0.9)

    print("\nTraining Complete! Multi-model predictions generated.")

    # Package all trained models into a dictionary for the .pkl export
    trained_models_dict = {
        'LGBM_P50': lgbm_p50,
        'LGBM_P90': lgbm_p90,
        'RandomForest_P50': rf_p50,
        'IsolationForest': iso_forest,
        'Survival_AFT': aft_model,
        'Survival_Scaler': surv_scaler
    }

    return test_data, trained_models_dict

# --- Execution Block ---
if __name__ == "__main__":
    train_path = os.path.join("..", "Data", "Processed", "train_data.csv")
    test_path = os.path.join("..", "Data", "Processed", "test_data.csv")
    output_path = os.path.join("..", "Data", "Processed", "Phase3_Advanced_Predictions.csv")

    # --- NEW: Defined the Models folder path ---
    model_save_path = os.path.join("..", "models", "final_models.pkl")

    try:
        train_data = pd.read_csv(train_path)
        test_data = pd.read_csv(test_path)

        target_col = 'Calculated_Actual_Days' if 'Calculated_Actual_Days' in train_data.columns else 'Actual_Transit_Days'
        features = ['Pickup_Day_of_Week', 'Pickup_Month', 'Is_Weekend_Pickup', 'orig_cntry_cd', 'dest_cntry_cd', 'dest_pstl_cd', 'Lane_Historical_Avg', 'Postal_Rolling_7D_Delay_Rate']
        if 'Cluster_ID' in train_data.columns: features.append('Cluster_ID')

        # 1. Run Cross Validation
        cross_validate_p50_model(train_data, features, target_col, n_splits=5)

        # 2. Train Models and get predictions
        results_df, trained_models = evaluate_all_models(train_data, test_data, features, target_col)

        # 3. Save the predictions for Phase 4
        results_df.to_csv(output_path, index=False)
        print(f"Saved advanced predictions to: {output_path}")

        # --- NEW: Create the folder and save the .pkl file ---
        os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
        joblib.dump(trained_models, model_save_path)

        print(f"✅ Successfully exported all trained AI models to: {model_save_path}")
        print("Ready for deployment & Phase 4 Business Logic!")

    except FileNotFoundError:
        print("[ERROR] Could not find training data. Run Phase 2 preprocessing first.")



--- Running 5-Fold Time-Series Cross-Validation (P50 Baseline) ---
 Fold 1: Mean Absolute Error = 1.96 days
 Fold 2: Mean Absolute Error = 2.03 days
 Fold 3: Mean Absolute Error = 1.89 days
 Fold 4: Mean Absolute Error = 2.31 days
 Fold 5: Mean Absolute Error = 2.06 days
 => Average CV Error: 2.05 days (±0.14 days)
-----------------------------------------------------------------------

Training Multi-Model Pipeline...
 - Training LightGBM...
 - Training Isolation Forest (Anomaly Detection)...
   -> Fitting Base RF Model (Median Target)...
   -> Fitting Isolation Forest (Finding Statistical Outliers)...
 - Training Survival Analysis (Weibull AFT)...

Training Complete! Multi-model predictions generated.
Saved advanced predictions to: ..\Data\Processed\Phase3_Advanced_Predictions.csv
✅ Successfully exported all trained AI models to: ..\models\final_models.pkl
Ready for deployment & Phase 4 Business Logic!


C:\Users\pimen\Documents\AI and ML\Capstone\.venv\Lib\site-packages\lifelines\fitters\__init__.py:2090: ApproximationWarning: The Hessian was not invertible. We will instead approximate it using the pseudo-inverse.

It's advisable to not trust the variances reported, and to be suspicious of the fitted parameters too.

Some ways to possible ways fix this:

0. Are there any lifelines warnings outputted during the `fit`?
1. Does a particularly large variable need to be centered to 0?
2. Inspect your DataFrame: does everything look as expected? Do you need to add/drop a constant (intercept) column?
3. Is there high-collinearity in the dataset? Try using the variance inflation factor (VIF) to find redundant variables.
4. Trying adding a small penalizer (or changing it, if already present). Example: `WeibullAFTFitter(penalizer=0.01).fit(...)`.
5. Are there any extreme outliers? Try modeling them or dropping them to see if it helps convergence.

  warnings.warn(warning_text, exceptions.Approx